In [2]:
!pip install -q transformers datasets torch accelerate evaluate scikit-learn

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
import evaluate

In [6]:
# 1. Define your exact storage location paths
EXPORT_DIR = "/content/drive/MyDrive/AIE_Project/intent_classification_datasets/"
train_path = f"{EXPORT_DIR}train_cleaned_balanced.csv"
val_path = f"{EXPORT_DIR}val_cleaned.csv"

# 2. Use Hugging Face to load the separate split files directly into a unified object
raw_datasets = load_dataset('csv', data_files={
    'train': train_path,
    'validation': val_path
})


# 3. Dynamically extract the unique text categories for labeling consistency
# We read from the validation set because it preserves your natural, un-duplicated categories
df_val_reference = pd.read_csv(val_path)
unique_labels = sorted(df_val_reference["label"].unique().tolist())

# 4. Print structured metrics to confirm the files match perfectly
print("==========================================================")
print("             DATA SPLIT LOAD REPORT                      ")
print("==========================================================\n")
print(f"✅ Total Unique Classes Loaded: {len(unique_labels)}")
print(f" -> Balanced Training Set Size : {len(raw_datasets['train'])} rows")
print(f" -> Natural Validation Set Size: {len(raw_datasets['validation'])} rows")
print(f"\nDetected Classes: {unique_labels}")


             DATA SPLIT LOAD REPORT                      

✅ Total Unique Classes Loaded: 9
 -> Balanced Training Set Size : 4716 rows
 -> Natural Validation Set Size: 469 rows

Detected Classes: ['complaint', 'general_question', 'logistics', 'payment', 'product_inquiry', 'refund', 'self_harm_or_suicide_risk', 'technical_support', 'unknown']


In [12]:
# ========================================================
# CELL 4 FIX: DYNAMIC TEXT TO INT LABEL CONVERSION
# ========================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
import evaluate
import numpy as np

# 1. Base model architecture selection
MODEL_NAME = "distilbert-base-uncased"

# 2. Build explicit mappings from the unique labels list (defined in cell 3)
id2label_clean = {str(idx): label for idx, label in enumerate(unique_labels)}
label2id_clean = {label: idx for idx, label in enumerate(unique_labels)}

# 3. Label Mapping Function to convert string labels into numeric integers
def encode_labels(example):
    # Converts string text label e.g., "refund" to 5
    example["label"] = label2id_clean[example["label"]]
    return example

# Apply label integer conversion to your train and validation sets
encoded_datasets = raw_datasets.map(encode_labels)

# 4. Load the tokenizer matching the core encoder model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Truncate and pad text strings to ensure uniform input length matrix dimensions
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Tokenize the data rows using our integer-mapped dataset
tokenized_datasets_raw = encoded_datasets.map(tokenize_function, batched=True)

# 5. Explicitly remove the raw 'text' string column so PyTorch doesn't try to tensor-ize it
columns_to_remove = [col for col in tokenized_datasets_raw["train"].column_names if col not in ["label", "input_ids", "attention_mask"]]
tokenized_datasets = tokenized_datasets_raw.remove_columns(columns_to_remove)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 6. Instantiate the architecture with explicit dynamic class configurations
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label_clean,
    label2id=label2id_clean
)

# 7. Connect standard metric logging evaluation frameworks
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

print(f"✅ Tokenizer and Model successfully initialized for {len(unique_labels)} classes!")
print(f"Kept features for training tensor conversion: {tokenized_datasets['train'].column_names}")


Map:   0%|          | 0/4716 [00:00<?, ? examples/s]

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

Map:   0%|          | 0/4716 [00:00<?, ? examples/s]

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Tokenizer and Model successfully initialized for 9 classes!
Kept features for training tensor conversion: ['label', 'input_ids', 'attention_mask']


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/final_intent_model",
    learning_rate=3e-5,                  # Fast, stable fine-tuning learning rate
    per_device_train_batch_size=16,       # Scaled batch sizing for T4 GPU acceleration
    per_device_eval_batch_size=16,
    num_train_epochs=6,                  # Balanced iteration count over the larger dataset
    weight_decay=0.01,
    eval_strategy="epoch",               # Evaluate accuracy metrics after each training loop pass
    save_strategy="epoch",
    load_best_model_at_end=True,         # Track and reload optimal state weights automatically
    metric_for_best_model="accuracy",
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Execute model training
trainer.train()

# Permanently persist model weights and configurations cleanly to storage disk
model.save_pretrained("/content/final_intent_model")
tokenizer.save_pretrained("/content/final_intent_model")
print("\n🎉 Training Complete! Clean model saved to /content/final_intent_model")


Epoch,Training Loss,Validation Loss,Accuracy
1,0.011435,0.011896,0.997868
2,0.003049,0.013708,0.997868
3,0.001574,0.011939,0.997868
4,0.001078,0.011985,0.997868
5,0.000870,0.012211,0.997868
6,0.000811,0.012202,0.997868


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 Training Complete! Clean model saved to /content/final_intent_model


In [14]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Run model predictions across the 20% validation split dataset
validation_output = trainer.predict(tokenized_datasets["validation"])

# 2. Extract the predicted class IDs and actual ground-truth label IDs
predicted_class_ids = np.argmax(validation_output.predictions, axis=-1)
true_class_ids = validation_output.label_ids

# 3. Print the comprehensive classification matrix report
print("==========================================================")
print("             FINAL MODEL TRAINING PERFORMANCE REPORT      ")
print("==========================================================\n")

print(classification_report(
    true_class_ids,
    predicted_class_ids,
    target_names=unique_labels,
    digits=4
))


             FINAL MODEL TRAINING PERFORMANCE REPORT      

                           precision    recall  f1-score   support

                complaint     1.0000    1.0000    1.0000        41
         general_question     1.0000    1.0000    1.0000        55
                logistics     0.9881    1.0000    0.9940        83
                  payment     1.0000    1.0000    1.0000       112
          product_inquiry     1.0000    0.9722    0.9859        36
                   refund     1.0000    1.0000    1.0000        67
self_harm_or_suicide_risk     1.0000    1.0000    1.0000        27
        technical_support     1.0000    1.0000    1.0000        30
                  unknown     1.0000    1.0000    1.0000        18

                 accuracy                         0.9979       469
                macro avg     0.9987    0.9969    0.9978       469
             weighted avg     0.9979    0.9979    0.9979       469



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Gather true labels and model predictions across validation data
predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# 2. Compute matrix configuration layout
cm = confusion_matrix(true_labels, preds)

# 3. Plot chart map
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=unique_labels, yticklabels=unique_labels, cmap="Blues")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Intent Classification Confusion Matrix')
plt.show()


Production JSON Inference

In [15]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. Load the model directly from disk
model_path = "/content/final_intent_model"
saved_model = AutoModelForSequenceClassification.from_pretrained(model_path)
saved_tokenizer = AutoTokenizer.from_pretrained(model_path)
saved_model.eval()

def get_structured_inference(text, confidence_threshold=0.75):
    inputs = saved_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

    with torch.no_grad():
        outputs = saved_model(**inputs)
        logits = outputs.logits

    # Translate raw logits to clean probability percentages
    probabilities = F.softmax(logits, dim=-1).squeeze().tolist()

    intent_scores = []
    for idx, prob in enumerate(probabilities):
        # FIXED: Look up using integer keys (idx) instead of string keys (str(idx))
        # If your config file uses integers, idx works. If it uses strings, we use str(idx).
        # This safe get() function checks both to prevent any future KeyError.
        label_name = saved_model.config.id2label.get(idx) or saved_model.config.id2label.get(str(idx))
        intent_scores.append({"intent": label_name, "confidence": round(prob, 2)})

    # Sort intent priorities
    intent_scores = sorted(intent_scores, key=lambda x: x["confidence"], reverse=True)

    top_intent = intent_scores[0]["intent"]        # FIXED: Added missing index [0] to grab top dict item
    top_confidence = intent_scores[0]["confidence"]  # FIXED: Added missing index [0] to grab top dict item
    top_3_intents = intent_scores[:3]

    # Check if confidence meets our threshold criteria
    requires_human_review = True if top_confidence < confidence_threshold else False

    response_payload = {
        "predicted_intent": top_intent,
        "confidence": top_confidence,
        "top_3_intents": top_3_intents,
        "requires_human_review": requires_human_review
    }

    return json.dumps(response_payload, indent=4)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [16]:
# --- Execute Test Validation Call ---
sample_text = "I want to cancel my order and get a refund."
print("Inference Output:\n")
print(get_structured_inference(sample_text))

Inference Output:

{
    "predicted_intent": "refund",
    "confidence": 0.99,
    "top_3_intents": [
        {
            "intent": "refund",
            "confidence": 0.99
        },
        {
            "intent": "complaint",
            "confidence": 0.0
        },
        {
            "intent": "general_question",
            "confidence": 0.0
        }
    ],
    "requires_human_review": false
}


In [ ]:
# Zip the model directory from your Colab local path
!zip -r final_intent_model.zip /content/final_intent_model

In [18]:
# Copy the zip from the local runtime space straight into your Drive folder
!cp final_intent_model.zip /content/drive/MyDrive/AIE_Project/models/

print("✅ Saved to Google Drive! You can download it directly from your Drive dashboard.")


✅ Saved to Google Drive! You can download it directly from your Drive dashboard.
